# Model 2 - 양수 위험도 회귀 모델

모델 2는 `위험도 > 0`인 포인트만 사용해서 사고 발생 시 위험도 크기를 예측하는 회귀 모델입니다.

- 입력: 전처리된 환경 feature
- 학습 대상 행: `위험도 > 0`
- 타겟: `위험도_log1p`
- 출력: `pred_위험도_log1p`, `pred_위험도`
- 의미: `E(위험도 | 위험도 > 0, x)`

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

# 노트북을 프로젝트 루트 또는 notebooks 폴더 어디서 실행해도 경로가 맞도록 처리합니다.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

VENV_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
PYTHON = VENV_PYTHON if VENV_PYTHON.exists() else Path(sys.executable)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PYTHON: {PYTHON}")

In [ ]:
def make_tensorflow_env() -> dict[str, str]:
    """TensorFlow subprocess가 pip NVIDIA CUDA 라이브러리를 찾도록 환경변수를 구성합니다."""
    env = os.environ.copy()
    site_packages_candidates = list((PROJECT_ROOT / ".venv" / "lib").glob("python*/site-packages"))
    nvidia_lib_dirs = []
    for site_packages in site_packages_candidates:
        nvidia_root = site_packages / "nvidia"
        if nvidia_root.exists():
            nvidia_lib_dirs.extend(str(path) for path in nvidia_root.rglob("lib") if path.is_dir())

    existing_ld_path = env.get("LD_LIBRARY_PATH", "")
    ld_parts = [*nvidia_lib_dirs, "/usr/lib/wsl/lib"]
    if existing_ld_path:
        ld_parts.append(existing_ld_path)
    env["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys(part for part in ld_parts if part))
    return env


tensorflow_env = make_tensorflow_env()
gpu_check_code = """
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('Physical GPUs:', gpus)
if gpus:
    print('GPU를 사용할 수 있습니다.')
else:
    print('GPU를 사용할 수 없습니다. CPU로 실행됩니다.')
"""

result = subprocess.run(
    [str(PYTHON), "-c", gpu_check_code],
    cwd=PROJECT_ROOT,
    env=tensorflow_env,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("TensorFlow GPU 확인 subprocess가 실패했습니다.")

In [ ]:
required_paths = {
    "train": PROJECT_ROOT / "data" / "processed" / "original_train_train_preprocessed.csv",
    "val": PROJECT_ROOT / "data" / "processed" / "original_train_val_preprocessed.csv",
    "test": PROJECT_ROOT / "data" / "processed" / "original_train_test_preprocessed.csv",
    "preprocess_config": PROJECT_ROOT / "artifacts" / "preprocessors" / "original_train_preprocess_config.json",
    "train_script": PROJECT_ROOT / "scripts" / "train" / "train_positive_risk_regressor.py",
}

path_status = []
for name, path in required_paths.items():
    path_status.append(
        {
            "name": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else None,
        }
    )

path_status_frame = pd.DataFrame(path_status)
display(path_status_frame)

missing = [row["path"] for row in path_status if not row["exists"]]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}")

In [ ]:
split_paths = {
    "train": required_paths["train"],
    "val": required_paths["val"],
    "test": required_paths["test"],
}

split_summary = []
for split_name, path in split_paths.items():
    frame = pd.read_csv(path, usecols=["위험도"])
    positive_count = int((frame["위험도"] > 0).sum())
    split_summary.append(
        {
            "split": split_name,
            "rows": len(frame),
            "positive_rows": positive_count,
            "positive_ratio": positive_count / len(frame),
        }
    )

display(pd.DataFrame(split_summary))

In [ ]:
# 학습 설정입니다. 빠른 테스트가 필요하면 LIMIT_ROWS 값을 5000 정도로 지정하십시오.
RUN_NAME = "mlp_positive_risk_regressor_notebook"
EPOCHS = 100
BATCH_SIZE = 1024
DEVICE = "auto"  # "auto", "gpu", "cpu"
LIMIT_ROWS = None  # 예: 5000

model_path = PROJECT_ROOT / "artifacts" / "models" / f"{RUN_NAME}.keras"
metrics_path = PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAME}_metrics.json"
history_path = PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAME}_history.csv"
predictions_path = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAME}_test_predictions.csv"

run_config = {
    "run_name": RUN_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "device": DEVICE,
    "limit_rows": LIMIT_ROWS,
    "model_path": str(model_path.relative_to(PROJECT_ROOT)),
    "metrics_path": str(metrics_path.relative_to(PROJECT_ROOT)),
    "history_path": str(history_path.relative_to(PROJECT_ROOT)),
    "predictions_path": str(predictions_path.relative_to(PROJECT_ROOT)),
}

run_config

In [ ]:
cmd = [
    str(PYTHON),
    str(PROJECT_ROOT / "scripts" / "train" / "train_positive_risk_regressor.py"),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--verbose", "2",
    "--model-path", str(model_path),
    "--metrics-path", str(metrics_path),
    "--history-path", str(history_path),
    "--predictions-path", str(predictions_path),
]
if LIMIT_ROWS is not None:
    cmd.extend(["--limit-rows", str(LIMIT_ROWS)])

print("실행 명령:")
print(" ".join(cmd))

# Keras verbose=2 설정으로 batch 진행 막대는 숨기고 epoch별 결과만 확인합니다.
process = subprocess.Popen(
    cmd,
    cwd=PROJECT_ROOT,
    env=tensorflow_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"모델 2 학습 실패: exit code {return_code}")

In [ ]:
with metrics_path.open("r", encoding="utf-8") as file:
    metrics = json.load(file)

print("split 행 수")
display(pd.DataFrame([metrics["raw_split_rows"]]).assign(kind="raw"))
display(pd.DataFrame([metrics["positive_split_rows"]]).assign(kind="positive_only"))

print("log1p scale 회귀 지표")
display(pd.DataFrame([metrics["log_scale"]]))

print("원래 위험도 scale 회귀 지표")
display(pd.DataFrame([metrics["risk_scale"]]))

print("위험도 threshold count")
display(pd.DataFrame([metrics["risk_threshold_counts"]]))

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv(history_path)
display(history.tail())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
history[["loss", "val_loss"]].plot(ax=axes[0], title="Huber Loss")
history[["mae", "val_mae"]].plot(ax=axes[1], title="MAE - log1p scale")
history[["rmse", "val_rmse"]].plot(ax=axes[2], title="RMSE - log1p scale")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout()
plt.show()

In [ ]:
predictions = pd.read_csv(predictions_path)
display(predictions.head())

print("예측 파일 크기:", predictions.shape)
print("실제 위험도 요약")
display(predictions["위험도"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame().T)
print("예측 위험도 요약")
display(predictions["pred_위험도"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame().T)

In [ ]:
sample = predictions.sample(min(3000, len(predictions)), random_state=42)
ax = sample.plot.scatter(
    x="위험도",
    y="pred_위험도",
    alpha=0.35,
    figsize=(6, 6),
    title="Actual vs predicted risk - positive-risk test subset",
)
limit = max(sample["위험도"].max(), sample["pred_위험도"].max())
ax.plot([0, limit], [0, limit], color="red", linewidth=1)
ax.set_xlabel("actual 위험도")
ax.set_ylabel("predicted 위험도")
plt.show()

In [ ]:
top_actual = predictions.sort_values("위험도", ascending=False).head(30)
display(top_actual[["POINT_ID", "위도", "경도", "위험도", "pred_위험도", "위험도_log1p", "pred_위험도_log1p"]])

top_predicted = predictions.sort_values("pred_위험도", ascending=False).head(30)
display(top_predicted[["POINT_ID", "위도", "경도", "위험도", "pred_위험도", "위험도_log1p", "pred_위험도_log1p"]])